In [1]:
# 1) Imports
import os, json, torch
import pandas as pd
from pathlib import Path
from datetime import datetime
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    DataCollatorForLanguageModeling, TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


c:\Users\Muham\OneDrive\Desktop\LoRA_FT\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2) Config
BASE_DIR   = Path(r"C:\Users\Muham\OneDrive\Desktop\LoRA_FT")
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"          # or "Qwen/Qwen2-1.5B-Instruct"
TRAIN_SIZE = 20000                                # scaled up for longer training
EPOCHS     = 6                                    # target ~6–7 hours on RTX 3050
MAX_LEN    = 256
BATCH_SIZE = 1
GRAD_ACC   = 16                                   # higher effective batch size
LR         = 1e-4                                 # smaller LR for stability

# unique output folder (no overwrite)
RUN_TAG    = datetime.now().strftime("%Y%m%d-%H%M%S")
output_dir = BASE_DIR / "outputs" / f"{MODEL_NAME.split('/')[-1].lower()}_{RUN_TAG}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Saving run to: {output_dir}")


Saving run to: C:\Users\Muham\OneDrive\Desktop\LoRA_FT\outputs\qwen2-0.5b-instruct_20251030-180847


In [3]:
# 3) Load data
twosides = pd.read_csv(BASE_DIR / "TwoSidesData.csv", low_memory=False)
offsides = pd.read_csv(BASE_DIR / "OffSidesData.csv", low_memory=False)

# sample
twosides_s = twosides.dropna().sample(min(TRAIN_SIZE//2, len(twosides)), random_state=42)
offsides_s = offsides.dropna().sample(min(TRAIN_SIZE//2, len(offsides)), random_state=42)
print(f"TwoSides: {len(twosides)} | OffSides: {len(offsides)}")
print(f"Sampled: TwoSides {len(twosides_s)} + OffSides {len(offsides_s)} = {len(twosides_s)+len(offsides_s)}")


TwoSides: 1048575 | OffSides: 848575
Sampled: TwoSides 10000 + OffSides 10000 = 20000


In [4]:
# 4) Build instruction-style texts
texts = []
for _, r in twosides_s.iterrows():
    d1 = str(r["drug_1_concept_name"]).strip()
    d2 = str(r["drug_2_concept_name"]).strip()
    cond = str(r["condition_concept_name"]).strip()
    texts.append(f"### Question:\nWhat adverse event might occur when taking {d1} and {d2} together?\n\n### Answer:\nWhen {d1} and {d2} are taken together, they may cause {cond}.")

for _, r in offsides_s.iterrows():
    d = str(r["drug_concept_name"]).strip()
    cond = str(r["condition_concept_name"]).strip()
    texts.append(f"### Question:\nWhat adverse event is associated with {d}?\n\n### Answer:\nThe drug {d} is associated with {cond}.")

print(f"Training samples: {len(texts)}")


Training samples: 20000


In [5]:
# 5) Tokenizer + 4-bit base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
model.gradient_checkpointing_enable()  # save VRAM for long runs
model = prepare_model_for_kbit_training(model)
print("Model + tokenizer ready")


Model + tokenizer ready


In [6]:
# 6) LoRA setup (targets cover Qwen/TinyLlama projections)
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [7]:
# 7) Tokenize dataset
def tokenize_fn(batch):
    enc = tokenizer(
        batch["text"], truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors=None
    )
    enc["labels"] = enc["input_ids"].copy()
    return enc

dataset = Dataset.from_dict({"text": texts})
tok_ds = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("Tokenization complete")


Map: 100%|██████████| 20000/20000 [00:05<00:00, 3425.24 examples/s]

Tokenization complete


In [8]:
# 8) Training
args = TrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    fp16=True,
    logging_steps=20,
    save_steps=500,
    save_total_limit=3,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_ds,
    data_collator=collator,
)
print("🏃 Training...")
trainer.train()
print("✅ Training complete")


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


🏃 Training...


c:\Users\Muham\OneDrive\Desktop\LoRA_FT\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,2.855900
40,2.703800
60,2.392100
80,1.846200
100,1.367800
120,0.973000
140,0.866800
160,0.830100
180,0.818700
200,0.785500


c:\Users\Muham\OneDrive\Desktop\LoRA_FT\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\Muham\OneDrive\Desktop\LoRA_FT\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users

✅ Training complete


In [9]:
# 9) Save adapter + tokenizer
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Saved adapter & tokenizer to: {output_dir}")


Saved adapter & tokenizer to: C:\Users\Muham\OneDrive\Desktop\LoRA_FT\outputs\qwen2-0.5b-instruct_20251030-180847


In [10]:
# 10) Quick test the fine-tuned model
model.eval()

test_prompts = [
    "What adverse event might occur when taking aspirin and warfarin together?",
    "What adverse event is associated with metformin?",
    "What adverse event might occur when taking ibuprofen and naproxen together?"
]

print("=" * 60)
print("🧪 QUICK TESTING")
print("=" * 60)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nTest {i}:")
    print(f"Question: {prompt}")
    print("-" * 60)
    
    formatted_prompt = f"### Question:\n{prompt}\n\n### Answer:\n"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_response.split("### Answer:")[-1].strip()
    print(f"Answer: {answer}")

print("\n" + "=" * 60)


🧪 QUICK TESTING

Test 1:
Question: What adverse event might occur when taking aspirin and warfarin together?
------------------------------------------------------------
Answer: When aspirin and warfarin are taken together, they may cause Blood pressure diastolic increased. It is correct. Explanation: When aspirin and warfarin are taken together, they may cause Blood pressure systolic increased. So the answer is incorrect.

Test 2:
Question: What adverse event is associated with metformin?
------------------------------------------------------------
Answer: The drug metformin is associated with Ovarian mass. It is associated with Hepatic function abnormal. A significant number of patients are at increased risk for venous thrombosis if they were taking Metformin and leflunomide together. They may experience a blood creatinine decreased. This is a potential drug reaction. It is associated with Blood bilirubin abnormal. A significant number of patients are at increased risk for venous thr